# htd second model

Llama-3.2-3B-Instruct through the same pipeline: SFT with the Tiny Aya data and recipe (seed 1), then base and SFT on test, held-out and control with the same generation settings.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    exact = [p for p in hits if "datasets/rockerleo/htd-code" in str(p)]
    hits = exact or hits
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted. input: " + str(sorted(str(p) for p in Path('/kaggle/input').glob('*'))))
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
SRC = find_repo()
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("code root:", SRC)
for rel, marker in {"scripts/train.py": "set_seed(a.seed)", "src/htd/backends.py": "def stop_token_ids", "kaggle/second_model/second_model.ipynb": "LLAMA"}.items():
    assert marker in (WORK / rel).read_text(encoding="utf-8"), f"stale code dataset: {rel} lacks {marker!r}"
print("code markers ok")

def find_dir(marker, parent_depth):
    hits = [p for p in Path("/kaggle/input").rglob(marker) if p.exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit(f"{marker} not mounted")
    p = hits[0]
    for _ in range(parent_depth):
        p = p.parent
    return p

def read_token():
    for p in Path("/kaggle/input").rglob("hf_token.txt"):
        return p.read_text().strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

if not os.environ.get("HF_TOKEN"):
    tok = read_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ.setdefault("HUGGINGFACE_API_KEY", tok)
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets (private) or add a Kaggle secret HF_TOKEN"

FIRE = "CohereLabs/tiny-aya-fire"
GLOBAL = "CohereLabs/tiny-aya-global"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def assert_adapter(name):
    p = WORK / "outputs" / name
    cfg = p / "adapter_config.json"
    weights = p / "adapter_model.safetensors"
    assert cfg.exists(), f"{name}: adapter_config.json missing"
    assert weights.exists() and weights.stat().st_size > 1_000_000, f"{name}: adapter weights missing or empty"
    print(f"{name}: adapter ok")


In [ ]:
pairs_src = find_dir("dpo_train.jsonl", 1)
dst = WORK / "data" / "generated" / "pairs"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(pairs_src, dst)
print("pairs:", sorted(p.name for p in dst.iterdir()))
n_sft = sum(1 for x in open(dst / "sft_train.jsonl") if x.strip())
assert n_sft == 858, f"expected 858 SFT rows (pairs v2), got {n_sft}"

In [ ]:
# Same library versions as sft_lr1e4 (htd-train v6), so the seed is the only change.
run([sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.20.2", "trl==0.29.1", "peft==0.21.2", "bitsandbytes>=0.45", "accelerate>=1.0"])
# PEFT refuses an incompatible torchao when loading an adapter for generation (eval v1 failure).
run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
assert importlib.util.find_spec("torchao") is None, "torchao is still installed"

In [ ]:
import importlib.metadata as md
versions = {p: md.version(p) for p in ("transformers", "trl", "peft", "bitsandbytes", "torch")}
print(versions)
assert (versions["transformers"], versions["trl"], versions["peft"]) == ("5.20.2", "0.29.1", "0.21.2"), versions

In [ ]:
# Second base model, same data and SFT recipe as Tiny Aya (seed 1, seeded LoRA init).
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
run([sys.executable, "scripts/train.py", "--method", "sft", "--base", LLAMA,
     "--data-dir", "data/generated/pairs", "--output", "outputs/llama_sft_s1",
     "--epochs", "1", "--lr", "1e-4", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8", "--seed", "1"])
assert_adapter("llama_sft_s1")
meta = json.loads((WORK / "outputs" / "llama_sft_s1" / "train_meta.json").read_text())
assert meta["base"] == LLAMA and meta["args"]["seed"] == 1 and meta["train"]["rows"] == 858, meta

In [ ]:
for split in ("test", "heldout", "control_test"):
    run([sys.executable, "scripts/build_pilot.py", "--split", split,
         "--n-per-script", "20", "--out", f"data/generated/{split}_dialogues.jsonl"])

def ids(path):
    return {json.loads(line)["dialogue_id"] for line in Path(path).read_text().splitlines() if line.strip()}

In [ ]:
# Same generation settings as kaggle/eval: plain prompt, greedy, 384 new tokens, batch 8.
TAG = {"test": "test", "heldout": "heldout", "control_test": "control"}
Path("outputs/eval_llama").mkdir(parents=True, exist_ok=True)
for label, adapter in (("llama_base", None), ("llama_sft", str(WORK / "outputs" / "llama_sft_s1"))):
    for split in ("test", "heldout", "control_test"):
        out = f"outputs/eval_llama/{label}_{TAG[split]}.jsonl"
        cmd = [sys.executable, "scripts/run_pilot.py", "--dialogues", f"data/generated/{split}_dialogues.jsonl",
               "--model", LLAMA, "--model-name", label, "--variant", "plain", "--backend", "hf",
               "--device", "cuda", "--batch-size", "8", "--max-new-tokens", "384", "--out", out]
        if adapter:
            cmd += ["--adapter", adapter]
        run(cmd)
        assert ids(out) == ids(f"data/generated/{split}_dialogues.jsonl"), out
print(sorted(p.name for p in Path("outputs/eval_llama").iterdir()))

In [ ]:
E = "outputs/eval_llama"
for split, tag in (("test", "test"), ("heldout", "heldout"), ("control", "control")):
    run([sys.executable, "scripts/score_run.py", f"{E}/llama_base_{tag}.jsonl", f"{E}/llama_sft_{tag}.jsonl",
         "--expect", "60", "--out", f"{E}/summary_{tag}.json"])
for tag in ("test", "heldout"):
    run([sys.executable, "scripts/compare.py", f"{E}/llama_base_{tag}.jsonl", f"{E}/llama_sft_{tag}.jsonl",
         "--out", f"{E}/cmp_llama_sft_{tag}.json"])
print(json.load(open(f"{E}/cmp_llama_sft_test.json"))["significance"])